Distance to The Median Voter

In [8]:
import numpy as np
import seaborn as sns
import os
from scipy.stats import kurtosis, skew
import pandas as pd

from rcv_distribution import *
from MDS_analysis import *
from voting_rules import *
from consistency import *
from null_elections import *

In [33]:
def median_voter(distribution):
    return np.median(x)

In [34]:
directory = "dataverse_files_2025"
election_table = pd.read_csv("election_table.csv")

In [38]:
for filename in os.listdir(directory):
    try:
        meta = "null_elections/" + filename
        df = pd.read_csv(meta, usecols=["candidate", "position"])
        df = df.dropna(subset=["candidate", "position"])
        df["candidate"] = df["candidate"].astype(str).str.strip()

        pmin, pmax = df["position"].min(), df["position"].max()
        den = pmax - pmin
        df["position_norm"] = (df["position"] - pmin) / den if den > 0 else 0.0

        # if duplicates exist, keep the first row per candidate
        normalized_distances = (
            df.drop_duplicates("candidate", keep="first")
            .set_index("candidate")["position_norm"]
            .to_dict()
        )
        meta = "np_data_new/" + filename[0: -4] + ".npy"
        dist = np.load(meta)
        dist_min, dist_max = np.min(dist), np.max(dist)
        den = dist_max - dist_min
        dist = (dist - dist_min) / den if den > 0 else 0.0 
        mvoter = median_voter(dist)


        irv_winner = election_table.loc[election_table['filename']==filename, 'irv_winner'].iloc[0]
        condorcet_winner = election_table.loc[election_table['filename']==filename, 'condorcet_winner'].iloc[0]

        irv_winner_position = normalized_distances[irv_winner]
        condorcet_winner_position = normalized_distances[condorcet_winner]

        dist_irv = abs(mvoter - irv_winner_position)
        dist_condorcet = abs(mvoter - condorcet_winner_position)

        irv_condorcet_distance = abs(irv_winner_position - condorcet_winner_position)

        median_voter_preference, median_voter_preference_position = min(
            normalized_distances.items(),
            key=lambda item: abs(item[1] - mvoter)
        )
        

        election_table.loc[election_table['filename']==filename, 'median_voter_position'] = mvoter
        election_table.loc[election_table['filename']==filename, 'irv_median_voter_distance'] = dist_irv
        election_table.loc[election_table['filename']==filename, 'condorcet_median_voter_distance'] = dist_condorcet
        election_table.loc[election_table['filename']==filename, 'median_voter_preference'] = median_voter_preference
        election_table.loc[election_table['filename']==filename, 'median_voter_preference_position'] = median_voter_preference_position

    except Exception as e:
        print(filename, " threw ", e)


        

Alaska_11052024_StateHouseD17.csv  threw  [Errno 2] No such file or directory: 'null_elections/Alaska_11052024_StateHouseD17.csv'
Alaska_11052024_StateHouseD2.csv  threw  [Errno 2] No such file or directory: 'null_elections/Alaska_11052024_StateHouseD2.csv'
Alaska_11052024_StateHouseD24.csv  threw  [Errno 2] No such file or directory: 'null_elections/Alaska_11052024_StateHouseD24.csv'
Alaska_11052024_StateHouseD25.csv  threw  [Errno 2] No such file or directory: 'null_elections/Alaska_11052024_StateHouseD25.csv'
Alaska_11052024_StateHouseD26.csv  threw  [Errno 2] No such file or directory: 'null_elections/Alaska_11052024_StateHouseD26.csv'
Alaska_11052024_StateHouseD29.csv  threw  [Errno 2] No such file or directory: 'null_elections/Alaska_11052024_StateHouseD29.csv'
Alaska_11052024_StateHouseD3.csv  threw  [Errno 2] No such file or directory: 'null_elections/Alaska_11052024_StateHouseD3.csv'
Alaska_11052024_StateHouseD33.csv  threw  [Errno 2] No such file or directory: 'null_elections

In [40]:
election_table.to_csv("election_table(2).csv")

In [27]:
irv_winner = election_table.loc[election_table['filename'].eq(filename), 'irv1'].iloc[0]
condorcet_winner = election_table.loc[election_table['filename'].eq(filename), 'condorcet1'].iloc[0]

In [6]:
csv = "dataverse_files/Alaska_08162022_HouseofRepresentativesSpecial.csv"

ballots, candidates = parse_election_data(csv)
election = voting_rules(ballots, candidates)
irv = election.irv()
condorcet = election.condorcet()

In [7]:
print (irv, " ", condorcet)

('Peltola, Mary S.', {'round_1': {'Peltola, Mary S.': 75879, 'Palin, Sarah': 59060, 'Begich, Nick': 53913}, 'majority': {'Peltola, Mary S.': 91386, 'Palin, Sarah': 86258}})   Begich, Nick


In [11]:
csv = "null_elections/Alaska_08162022_HouseofRepresentativesSpecial.csv"
df = pd.read_csv(csv, usecols=["candidate", "position"])
df = df.dropna(subset=["candidate", "position"])
df["candidate"] = df["candidate"].astype(str).str.strip()

pmin, pmax = df["position"].min(), df["position"].max()
den = pmax - pmin
df["position_norm"] = (df["position"] - pmin) / den if den > 0 else 0.0

# if duplicates exist, keep the first row per candidate
normalized_distances = (
    df.drop_duplicates("candidate", keep="first")
      .set_index("candidate")["position_norm"]
      .to_dict()
)

print(normalized_distances)

{'Peltola, Mary S.': 0.0, 'Begich, Nick': 0.5850435735984745, 'Palin, Sarah': 1.0}


In [28]:
print(abs(mvoter - normalized_distances[irv_winner]))
print(abs(mvoter - normalized_distances[condorcet]))

0.4435357733051216
0.14150780029335286


In [41]:
import numpy as np
from sklearn.cluster import KMeans
from pathlib import Path
import pandas as pd

def _cohen_d_from_two_groups(g1: np.ndarray, g2: np.ndarray) -> float:
    """Cohen's d with pooled SD using ddof=1."""
    n1, n2 = g1.size, g2.size
    if n1 == 0 or n2 == 0:
        return np.nan
    mu1, mu2 = g1.mean(), g2.mean()
    s1 = g1.std(ddof=1) if n1 > 1 else 0.0
    s2 = g2.std(ddof=1) if n2 > 1 else 0.0
    denom_df = n1 + n2 - 2
    if denom_df <= 0:
        return np.nan
    sp2 = ((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / denom_df
    sp = np.sqrt(sp2)
    if sp == 0:
        # If both groups have zero variance: d is 0 if means equal, inf otherwise
        return 0.0 if np.isclose(mu1, mu2) else np.inf
    return abs(mu1 - mu2) / sp

def bimodality_from_array(x, normalize=True, kmeans_inits=50, random_state=0):
    """
    Parameters
    ----------
    x : array-like
        1D sample from the distribution.
    normalize : bool
        If True, min-max normalize to [0,1] before clustering (safe for mixed scales).
    Returns
    -------
    d : float
        Cohen's d between the two k-means groups.
    meta : dict
        Useful diagnostics: n1, n2, mu1, mu2, s1, s2, sp, labels.
    """
    x = np.asarray(x).ravel()
    x = x[np.isfinite(x)]
    if x.size < 3:
        return np.nan, {"note": "too few points", "n": x.size}

    if normalize and np.ptp(x) > 0:
        x = (x - x.min()) / (x.max() - x.min())

    # 1D K-means split
    km = KMeans(n_clusters=2, n_init=kmeans_inits, random_state=random_state)
    labels = km.fit_predict(x[:, None])

    g1 = x[labels == 0]
    g2 = x[labels == 1]
    d = _cohen_d_from_two_groups(g1, g2)

    meta = {
        "n1": g1.size, "n2": g2.size,
        "mu1": g1.mean() if g1.size else np.nan,
        "mu2": g2.mean() if g2.size else np.nan,
        "s1": g1.std(ddof=1) if g1.size > 1 else 0.0,
        "s2": g2.std(ddof=1) if g2.size > 1 else 0.0,
        "sp": np.nan,  # filled below
        "labels": labels
    }
    # compute pooled SD for the meta dict
    n1, n2 = meta["n1"], meta["n2"]
    denom_df = n1 + n2 - 2
    if n1 > 0 and n2 > 0 and denom_df > 0:
        sp2 = ((n1 - 1) * meta["s1"]**2 + (n2 - 1) * meta["s2"]**2) / denom_df
        meta["sp"] = np.sqrt(sp2)
    return d, meta

def bimodality_from_npy(path, **kwargs):
    """Load a .npy file and return (d, meta)."""
    arr = np.load(path)
    return bimodality_from_array(arr, **kwargs)

def bimodality_for_folder(folder, pattern="*.npy", **kwargs):
    """Compute d for every .npy in a folder; returns a DataFrame."""
    rows = []
    for p in Path(folder).glob(pattern):
        d, meta = bimodality_from_npy(p, **kwargs)
        rows.append({"file": p.name, "d": d, **meta})
    return pd.DataFrame(rows).sort_values("d", ascending=False)


In [58]:
directory = "dataverse_files_2025"
election_table = pd.read_csv("election_table(2).csv")
for filename in os.listdir(directory):
    try:
        npy_file = filename[0: -4] + ".npy"
        bimodality, meta = bimodality_from_npy("np_data_new/" + npy_file)
        election_table.loc[election_table['filename']==filename, 'bimodality'] = bimodality
    except Exception as e:
        print(filename," threw ", e)




Alaska_11052024_StateHouseD17.csv  threw  [Errno 2] No such file or directory: 'np_data_new/Alaska_11052024_StateHouseD17.npy'
Alaska_11052024_StateHouseD2.csv  threw  [Errno 2] No such file or directory: 'np_data_new/Alaska_11052024_StateHouseD2.npy'
Alaska_11052024_StateHouseD24.csv  threw  [Errno 2] No such file or directory: 'np_data_new/Alaska_11052024_StateHouseD24.npy'
Alaska_11052024_StateHouseD25.csv  threw  [Errno 2] No such file or directory: 'np_data_new/Alaska_11052024_StateHouseD25.npy'
Alaska_11052024_StateHouseD26.csv  threw  [Errno 2] No such file or directory: 'np_data_new/Alaska_11052024_StateHouseD26.npy'
Alaska_11052024_StateHouseD29.csv  threw  [Errno 2] No such file or directory: 'np_data_new/Alaska_11052024_StateHouseD29.npy'
Alaska_11052024_StateHouseD3.csv  threw  [Errno 2] No such file or directory: 'np_data_new/Alaska_11052024_StateHouseD3.npy'
Alaska_11052024_StateHouseD33.csv  threw  [Errno 2] No such file or directory: 'np_data_new/Alaska_11052024_StateHo

In [59]:
election_table.to_csv("election_table.csv", index=False)

In [68]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ---- start from your DataFrame ----
df = election_table.copy()

# keep rows you want (optional, mirrors your plots)
df = df[(df['candidates'] >= 2) & (df['candidates'] <= 22)].copy()

# --- outcomes: transform for linear modeling ---
eps = 1e-6
df['gamma_clip']  = df['gamma'].clip(eps, 1 - eps)
df['logit_gamma'] = np.log(df['gamma_clip'] / (1 - df['gamma_clip']))       # for gamma
df['log_b']       = np.log(df['bimodality'] + eps)                          # for bimodality

# --- predictors: clean up categoricals ---
df['level']    = df['level'].astype(str).str.strip().str.upper().astype('category')
df['type']     = df['type'].astype(str).str.strip().astype('category')

# partisan has 4 values; keep all four as a factor
df['partisan'] = pd.Categorical(
    df['partisan']
      .astype('string')       # keeps missing as <NA>
      .str.strip()
      .str.upper()
      .replace({'DEM':'DP', 'DEMOCRAT':'DP',
                'REP':'RP', 'REPUBLICAN':'RP'}),
    categories=['NO','YES','DP','RP']           # set your desired order
)
df['partisan'] = df['partisan'].replace({'DEM':'DP','REP':'RP'})  # if any variants exist
df['partisan'] = df['partisan'].astype('category')

# --- choose reference/baseline levels (change if you prefer) ---
# level baseline: LOCAL; partisan baseline: NO; type baseline = its first category
f_gamma = "logit_gamma ~ C(level, Treatment(reference='LOCAL')) + C(type) + C(partisan, Treatment(reference='NO'))"
f_b     = "log_b       ~ C(level, Treatment(reference='LOCAL')) + C(type) + C(partisan, Treatment(reference='NO'))"

# --- fit with robust (HC3) SEs ---
res_g = smf.ols(f_gamma, data=df).fit(cov_type='HC3')
res_b = smf.ols(f_b,     data=df).fit(cov_type='HC3')

print("\n=== Coefficients: logit(gamma) ===")
print(res_g.summary())

print("\n=== Coefficients: log(bimodality) ===")
print(res_b.summary())

# ---- Overall significance by TERM (robust Wald tests) ----
print("\n=== TERM significance (robust Wald) for logit(gamma) ===")
print(res_g.wald_test_terms(skip_single=False))

print("\n=== TERM significance (robust Wald) for log(bimodality) ===")
print(res_b.wald_test_terms(skip_single=False))

# (Optional) classical Type-II ANOVA tables (non-robust) for a second view:
from statsmodels.stats.anova import anova_lm
print("\nType-II ANOVA (non-robust) for logit(gamma):")
print(anova_lm(res_g, typ=2))
print("\nType-II ANOVA (non-robust) for log(bimodality):")
print(anova_lm(res_b, typ=2))



=== Coefficients: logit(gamma) ===
                            OLS Regression Results                            
Dep. Variable:            logit_gamma   R-squared:                       0.231
Model:                            OLS   Adj. R-squared:                  0.217
Method:                 Least Squares   F-statistic:                     19.21
Date:                Thu, 14 Aug 2025   Prob (F-statistic):           1.41e-27
Time:                        00:08:32   Log-Likelihood:                -1489.0
No. Observations:                 498   AIC:                             2998.
Df Residuals:                     488   BIC:                             3040.
Df Model:                           9                                         
Covariance Type:                  HC3                                         
                                                        coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------

c:\Users\mahsh\anaconda3\lib\site-packages\statsmodels\base\model.py:1889: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
c:\Users\mahsh\anaconda3\lib\site-packages\statsmodels\base\model.py:1889: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
